In [0]:
import time
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql import types as T

dbutils.widgets.text("catalogo", "workspace")

catalogo = dbutils.widgets.get("catalogo")

spark.sql(f"USE CATALOG {catalogo}")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")


TABELA_CLIENTES = "silver.tb_clientes"
TABELA_PEDIDOS = "silver.fat_pedidos"
TABELA_TICKETS = "gold.gold_tickets"
TABELA_CLICKSTREAM = "silver.tb_clickstream"
TABELA_AVALIACOES ="silver.tb_avaliacoes"
TABELA_DESTINO_CLIENTES = "gold_cliente_360"

In [0]:
df_clientes = spark.table(TABELA_CLIENTES)
df_pedidos = spark.table(TABELA_PEDIDOS)
df_tickets = spark.table(TABELA_TICKETS)
df_clickstream = spark.table(TABELA_CLICKSTREAM)
df_avaliacoes = spark.table(TABELA_AVALIACOES)

In [0]:
spark.sql("SHOW TABLES IN silver").show(truncate=False)

In [0]:
spark.sql("SHOW TABLES IN gold").show(truncate=False)

In [0]:
df_clientes_gold = df_clientes.select(
    "id_cliente",
    F.concat_ws(
        " ",
        F.col("nome"),
        F.col("sobrenome")
    ).alias("nome"),

    "email",
    "telefone",
    "data_cadastro",
    "cidade",

    F.col("estado").alias("uf"),

    "origem"
)

In [0]:
df_pedidos_agg = df_pedidos.groupBy("id_cliente").agg(

    # quantidade total pedidos
    F.count("id_pedido").alias("qtd_pedidos_total"),

    # aprovados
    F.sum(
        F.when(
            F.col("status") == "Aprovado",
            1
        ).otherwise(0)
    ).alias("qtd_pedidos_aprovados"),

    # recusados
    F.sum(
        F.when(
            F.col("status") == "Recusado",
            1
        ).otherwise(0)
    ).alias("qtd_pedidos_recusados"),

    # reembolsados
    F.sum(
        F.when(
            F.col("status") == "Reembolsado",
            1
        ).otherwise(0)
    ).alias("qtd_pedidos_reembolsados"),

    # processando
    F.sum(
        F.when(
            F.col("status") == "Processando",
            1
        ).otherwise(0)
    ).alias("qtd_pedidos_processando"),

    # valor total gasto
    F.round(
        F.sum("valor_total"),
        2
    ).alias("valor_total_gasto"),

    # ticket médio
    F.round(
        F.avg("valor_total"),
        2
    ).alias("ticket_medio"),

    # primeiro pedido
    F.min("data_pedido").alias("data_primeiro_pedido"),

    # último pedido
    F.max("data_pedido").alias("data_ultimo_pedido")
)

In [0]:
df_tickets_agg = df_tickets.groupBy("id_cliente").agg(
    F.count("*").alias("qtd_tickets_total"),
    
    F.sum(
        F.when(F.col("status_ticket") == "Aberto", F.lit(1))
        .otherwise(F.lit(0))
    ).alias("qtd_tickets_abertos"),
    
    F.sum(
        F.when(F.col("status_ticket") == "Resolvido", F.lit(1))
        .otherwise(F.lit(0))
    ).alias("qtd_tickets_resolvidos")
)

In [0]:
df_avaliacoes.printSchema()

In [0]:
df_avaliacoes_agg = df_avaliacoes.groupBy("id_cliente").agg(

    F.count("id_avaliacao").alias("qtd_avaliacoes"),

    F.round(
        F.avg("nota_produto"),
        2
    ).alias("nota_media_dada"),

    F.round(
        F.avg("nota_nps"),
        2
    ).alias("nps_medio_avaliacoes_cliente")
)

display(df_avaliacoes_agg)

In [0]:
df_clickstream = df_clickstream.groupBy("id_cliente").agg(

    F.count("*").alias("qtd_eventos_clickstream"),

    F.mode("canal").alias("canal_preferido")
)

In [0]:
df_gold = (
    df_clientes_gold

    .join(df_pedidos_agg, on="id_cliente", how="left")

    .join(df_tickets_agg, on="id_cliente", how="left")

    .join(df_avaliacoes_agg, on="id_cliente", how="left")

    .join(df_clickstream, on="id_cliente", how="left")
)

In [0]:
df_gold = df_gold.fillna({

    # pedidos
    "qtd_pedidos_total": 0,
    "qtd_pedidos_aprovados": 0,
    "qtd_pedidos_recusados": 0,
    "qtd_pedidos_reembolsados": 0,
    "qtd_pedidos_processando": 0,

    # tickets
    "qtd_tickets_total": 0,
    "qtd_tickets_abertos": 0,
    "qtd_tickets_resolvidos": 0,

    # clickstream
    "qtd_eventos_clickstream": 0
})

In [0]:
df_gold = df_gold.withColumn(
    "segmento_ltv",

    F.when(F.col("valor_total_gasto") >= 5000, "Alto")

    .when(F.col("valor_total_gasto") >= 1000, "Medio")

    .otherwise("Baixo")
)

In [0]:
df_gold = df_gold.withColumn(
    "is_ativo_90d",

    F.datediff(
        F.current_date(),
        F.col("data_ultimo_pedido")
    ) <= 90
)

In [0]:
df_gold = df_gold.withColumn(
    "is_em_risco",

    F.coalesce(
        F.col("qtd_tickets_abertos"),
        F.lit(0)
    ) >= 3
)

In [0]:
df_gold_cliente_360 = df_gold.withColumn(
    "data_referencia_calculo",
    F.current_date()
)

In [0]:

df_gold_cliente_360.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABELA_DESTINO_CLIENTES)

print(f"Tabela de tickets salva com sucesso em: {TABELA_DESTINO_CLIENTES}")

display(df_gold_cliente_360.limit(5))

In [0]:

df_gold_cliente_360.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABELA_DESTINO_CLIENTES)

print(f"Tabela de tickets salva com sucesso em: {TABELA_DESTINO_CLIENTES}")

display(df_gold_cliente_360.limit(5))

In [0]:
def exportar_csv(df, nome_arquivo):

    caminho_saida = f"/Volumes/workspace/gold/exports/{nome_arquivo}"

    (
        df.coalesce(1)
        .write
        .mode("overwrite")
        .option("header", "true")
        .csv(caminho_saida)
    )

    print(f"CSV exportado com sucesso em: {caminho_saida}")

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS gold.exports
""")

In [0]:
exportar_csv(
    df_gold_cliente_360,
    "gold_cliente_360_csv"
)